<a href="https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My baseline rule assigns a score to each content page using observed historical signals.

Pages receive a higher score when they have:
- Low average position (higher ranking number)
- Low CTR
- Older content
- Many impressions

These pages are likely candidates for content refresh or optimization.

Reason codes:
- REFRESH_OLD_CONTENT
- LOW_CTR
- LOW_RANKING
- HIGH_IMPRESSIONS


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))


Rows: 30000
Columns: 44


## 2. Build the ranked queue (writes the CSV)

The baseline score combines historical signals available before making a recommendation.

Higher scores indicate higher priority for review.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Fill missing values
df["ctr"] = df["ctr"].fillna(0)
df["avg_position"] = df["avg_position"].fillna(100)
df["content_age_days"] = df["content_age_days"].fillna(0)
df["impressions_90d"] = df["impressions_90d"].fillna(0)

# Simple baseline score
df["baseline_score"] = (
    (100 - df["ctr"]) +
    df["avg_position"] +
    (df["content_age_days"] / 30) +
    (df["impressions_90d"] / 100)
)

# Reason code
def reason_code(row):
    if row["ctr"] < 1:
        return "LOW_CTR"
    elif row["avg_position"] > 20:
        return "LOW_RANKING"
    elif row["content_age_days"] > 365:
        return "REFRESH_OLD_CONTENT"
    else:
        return "HIGH_IMPRESSIONS"

df["reason_code"] = df.apply(reason_code, axis=1)

df["action"] = "Review Content"

ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

ranked[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]].head(10)


## 3. Top-20 review

The top 20 pages have the highest baseline scores.

These pages are recommended for manual review because they combine multiple observed signals indicating potential optimization opportunities.

The recommendations are decision-support only.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)

for i, row in top20.iterrows():
    print(f"Content ID: {row['content_id']}")
    print(f"Action: {row['action']}")
    print(f"Reason Code: {row['reason_code']}")
    print("Confidence: Medium")
    print("What would make it wrong: External changes such as search demand shifts or algorithm updates may explain the observed performance.")
    print("-" * 60)


## 4. Weak picks + leakage check

Some pages may receive high scores because of one strong signal while other signals are weak.

The baseline rule cannot capture complex relationships between multiple features.

No target labels, future performance fields, or recommendation outputs were used when creating the score.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Weak picks review")

weak = ranked[
    (ranked["ctr"] > 5) &
    (ranked["avg_position"] < 10)
].head(5)

print(weak[[
    "content_id",
    "ctr",
    "avg_position",
    "baseline_score"
]])

print("\nLeakage check")

leakage_columns = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

for col in leakage_columns:
    print(f"{col}: Present in dataset but NOT used in the baseline score.")



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.


In [2]:
import os

print(os.getcwd())


/content


In [3]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))


False


In [7]:
import os
import subprocess

REPO_URL = "https://github.com/Ahmed-coder874/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print(os.getcwd())


/content/flyrank-ml-internship


In [8]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
